# Entrenamiento Local — Aprendizaje Federado MNIST

Cada integrante del equipo corre este notebook sobre su **partición privada de datos**.
Solo los **pesos** del modelo entrenado se comparten con el paso de agregación.

**Pasos:**
1. Cargar la partición (generada con `split_data.py`)
2. Entrenar el ResNet-Mini por `N_EPOCHS` épocas
3. Revisar las curvas de aprendizaje y el reporte de clasificación
4. Guardar los pesos en `weights/client_<id>_weights.weights.h5`

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

from TheModel import build

print('TensorFlow', tf.__version__)

## Configuración
Cambia `CLIENT_ID` al número de partición asignado (0, 1, 2, …).

In [ ]:
CLIENT_ID   = 0          # <- cambiar por integrante (0, 1, 2, 3, 4 o 5)
N_EPOCHS    = 10
BATCH_SIZE  = 64
DATA_DIR    = Path('data_partitions')
WEIGHTS_DIR = Path('weights')

SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

## Cargar Partición de Datos

In [ ]:
data_path = DATA_DIR / f'client_{CLIENT_ID}_data.npz'
data = np.load(str(data_path))

x_train, y_train = data['x_train'], data['y_train']
x_test,  y_test  = data['x_test'],  data['y_test']

print(f'Client {CLIENT_ID} — {len(x_train)} training samples')
print(f'Class distribution: {np.bincount(y_train).tolist()}')
print(f'Test set size     : {len(x_test)}')

## Modelo

In [ ]:
model = build.build_it()
model.summary()

## Ciclo de Entrenamiento

In [ ]:
history = model.fit(
    x_train, y_train,
    epochs=N_EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.15,
    verbose=1,
)

## Curvas de Aprendizaje

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history['loss'],     label='Train', linewidth=2)
axes[0].plot(history.history['val_loss'], label='Val',   linewidth=2, linestyle='--')
axes[0].set_title(f'Client {CLIENT_ID} — Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Cross-Entropy Loss')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(history.history['accuracy'],     label='Train', linewidth=2)
axes[1].plot(history.history['val_accuracy'], label='Val',   linewidth=2, linestyle='--')
axes[1].set_title(f'Client {CLIENT_ID} — Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle(f'Local Training — Client {CLIENT_ID}', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(f'learning_curves_client_{CLIENT_ID}.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Figure saved: learning_curves_client_{CLIENT_ID}.png')

## Evaluación

In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f'Test Loss    : {test_loss:.4f}')
print(f'Test Accuracy: {test_acc:.4f}')

## Reporte de Clasificación

In [ ]:
y_pred = model.predict(x_test, verbose=0).argmax(axis=1)

print('Classification Report (Client', CLIENT_ID, ')\n')
print(classification_report(
    y_test, y_pred,
    target_names=[str(i) for i in range(10)]
))

## Matriz de Confusión

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=list(range(10)),
    cmap='Blues',
    ax=ax,
    colorbar=False,
)
ax.set_title(f'Confusion Matrix — Client {CLIENT_ID}')
plt.tight_layout()
plt.savefig(f'confusion_matrix_client_{CLIENT_ID}.png', dpi=120, bbox_inches='tight')
plt.show()

## Guardar Pesos

In [ ]:
WEIGHTS_DIR.mkdir(exist_ok=True)
weight_path = str(WEIGHTS_DIR / f'client_{CLIENT_ID}_weights.weights.h5')
model.save_weights(weight_path)
print(f'Weights saved → {weight_path}')